# 03 — Intent Classifier (Zero-Shot with Groq)

Zero-shot intent classification using a Groq-hosted LLM.

**5 intents:** `greeting` · `goodbye` · `gratitude` · `asking_mental_health_question` · `out_of_scope`

**Setup:** `export GROQ_API_KEY='gsk_...'` — free key at [console.groq.com](https://console.groq.com)  
Runs without a key too — falls back to keyword heuristics automatically.

In [ ]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import os, textwrap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from src.modules.intent_classifier import IntentClassifier, VALID_INTENTS, _parse_intent, _keyword_fallback

plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

api_key = os.getenv('GROQ_API_KEY', '')
mode = 'Groq LLM' if api_key else 'keyword fallback (no GROQ_API_KEY set)'
print(f'Mode: {mode}')
print(f'Valid intents: {VALID_INTENTS}')

## 1. Prompt Template

In [ ]:
prompt_path = ROOT / 'src' / 'prompts' / 'intent_classification.txt'
prompt = prompt_path.read_text()
# Show everything except the variable placeholder at the end
preview = prompt[:prompt.rfind('---')].strip()
print(preview)

## 2. Response Parser — Unit Tests

In [ ]:
# Test _parse_intent with every edge case the LLM might produce
parse_tests = [
    # (raw_response,                          expected_intent,                  expected_conf)
    ('greeting',                              'greeting',                       1.0),
    ('asking_mental_health_question',         'asking_mental_health_question',  1.0),
    ('out_of_scope',                          'out_of_scope',                   1.0),
    ('GREETING',                              'greeting',                       1.0),
    ('greeting.',                             'greeting',                       1.0),
    ('  goodbye  ',                           'goodbye',                        1.0),
    ('Intent: greeting',                      'greeting',                       0.7),
    ('The intent is asking_mental_health_question', 'asking_mental_health_question', 0.7),
    ('mental_health',                         'asking_mental_health_question',  0.7),
    ('thank_you',                             'gratitude',                      0.7),
    ('off_topic',                             'out_of_scope',                   0.7),
    ('unknown',                               'out_of_scope',                   0.7),
    ('completely_random_junk',                'out_of_scope',                   0.4),
    ('',                                      'out_of_scope',                   0.4),
]

print(f'  {"Raw response":<45}  {"Intent":<34}  Conf  OK')
print('-' * 95)
all_ok = True
for raw, exp_intent, exp_conf in parse_tests:
    intent, conf = _parse_intent(raw)
    ok = (intent == exp_intent) and (conf == exp_conf)
    all_ok = all_ok and ok
    mark = '✓' if ok else '✗'
    print(f'  {mark} {repr(raw):<43}  {intent:<34}  {conf:.1f}')
print(f'\nParser tests: {"ALL PASSED ✅" if all_ok else "FAILURES ❌"}')

## 3. Keyword Fallback — Unit Tests

In [ ]:
fallback_tests = [
    ('Hi there!',                                       'greeting'),
    ('Good morning, how are you?',                      'greeting'),
    ('Goodbye!',                                        'goodbye'),
    ('See you later, take care',                        'goodbye'),
    ('Thank you so much',                               'gratitude'),
    ('I really appreciate your help',                   'gratitude'),
    ('I feel really anxious about my exam',             'asking_mental_health_question'),
    ('How do I deal with depression?',                  'asking_mental_health_question'),
    ('I cannot sleep because of all the stress',        'asking_mental_health_question'),
    ('I feel so lonely and worthless',                  'asking_mental_health_question'),
    ('Tell me a joke',                                  'out_of_scope'),
    ('What is 2 + 2?',                                  'out_of_scope'),
]

correct = 0
print(f'  {"Text":<50}  {"Expected":<34}  {"Got":<34}  OK')
print('-' * 125)
for text, expected in fallback_tests:
    intent, conf = _keyword_fallback(text)
    ok = intent == expected
    correct += ok
    print(f'  {"✓" if ok else "✗"} {text:<48}  {expected:<34}  {intent:<34}')
print(f'\nKeyword fallback: {correct}/{len(fallback_tests)} correct')

## 4. Full Wrapper — 17-Query Test Suite

In [ ]:
TEST_CASES = [
    # Required by phase spec
    ('Hi there!',                                               'greeting'),
    ('I\'m anxious about work',                                'asking_mental_health_question'),
    ('Thanks for helping',                                      'gratitude'),
    ('See you later',                                           'goodbye'),
    ('Tell me a joke',                                          'out_of_scope'),
    # Additional coverage
    ('Good morning!',                                           'greeting'),
    ('Hey, how are you?',                                       'greeting'),
    ('Goodbye, take care',                                      'goodbye'),
    ('I appreciate your support so much',                       'gratitude'),
    ('How do I cope with depression?',                          'asking_mental_health_question'),
    ('I can\'t sleep, the stress is overwhelming',             'asking_mental_health_question'),
    ('I feel so lonely and worthless lately',                   'asking_mental_health_question'),
    ('What are some strategies for panic attacks?',             'asking_mental_health_question'),
    ('I haven\'t been able to get out of bed this week',       'asking_mental_health_question'),
    ('What is the capital of France?',                          'out_of_scope'),
    ('Can you write me a poem?',                                'out_of_scope'),
    ('What\'s 10 times 7?',                                    'out_of_scope'),
]

clf = IntentClassifier()
print(f'Model / mode: {clf.model if api_key else "keyword_fallback"}')
print()

results = []
correct = 0
print(f'  {"#":<3}  {"Text":<52}  {"Expected":<34}  {"Got":<34}  Conf  Mode')
print('-' * 145)
for i, (text, expected) in enumerate(TEST_CASES, 1):
    r = clf.classify(text)
    ok = r['intent'] == expected
    correct += ok
    results.append({**r, 'text': text, 'expected': expected, 'correct': ok})
    short_mode = r['model_used'].replace('llama-3.3-70b-versatile', 'llama-70b').replace('keyword_fallback','kw-fallback')
    print(f'  {"✓" if ok else "✗"} {i:<2}  {text[:50]:<52}  {expected:<34}  {r["intent"]:<34}  {r["confidence"]:.2f}  {short_mode}')

pct = correct / len(TEST_CASES) * 100
print(f'\nResult: {correct}/{len(TEST_CASES)} correct  ({pct:.0f}%)  {"✅ PASS" if correct == len(TEST_CASES) else "❌ check failures above"}')

## 5. Results Visualisation

In [ ]:
import pandas as pd
df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Accuracy per intent
acc_per = df.groupby('expected')['correct'].mean().reindex(VALID_INTENTS)
colors = ['#2ecc71' if v == 1.0 else '#e74c3c' for v in acc_per]
axes[0].bar(acc_per.index, acc_per.values, color=colors, edgecolor='white')
axes[0].set_ylim(0, 1.15)
axes[0].axhline(1.0, color='grey', linestyle='--', linewidth=0.8)
axes[0].set_title('Accuracy per intent', fontsize=12)
axes[0].tick_params(axis='x', rotation=25)
axes[0].set_ylabel('Accuracy')
for i, (intent, val) in enumerate(acc_per.items()):
    axes[0].text(i, val + 0.02, f'{val:.0%}', ha='center', fontsize=9)

# Confidence distribution
conf_colors = {'1.0': '#3498db', '0.7': '#f39c12', '0.6': '#95a5a6', '0.4': '#e74c3c'}
conf_counts = df['confidence'].round(1).astype(str).value_counts()
bar_colors = [conf_colors.get(k, '#bdc3c7') for k in conf_counts.index]
axes[1].bar(conf_counts.index, conf_counts.values, color=bar_colors, edgecolor='white')
axes[1].set_title('Confidence score distribution', fontsize=12)
axes[1].set_xlabel('Confidence'); axes[1].set_ylabel('Count')
patches = [mpatches.Patch(color=v, label=f'{k} — {"exact" if k=="1.0" else "fuzzy" if k=="0.7" else "kw-fallback" if k=="0.6" else "hard fallback"}')
           for k, v in conf_colors.items() if k in conf_counts.index.astype(str)]
axes[1].legend(handles=patches, fontsize=8)

plt.suptitle(f'Intent Classifier Results  ({correct}/{len(TEST_CASES)} correct)', fontsize=13)
plt.tight_layout(); plt.show()

## 6. Fallback Logic Documentation

In [ ]:
print('''
┌─────────────────────────────────────────────────────────────────┐
│              IntentClassifier — Fallback Logic                  │
├──────────────────────┬──────────────────────────────────────────┤
│ Scenario             │ Behaviour                                │
├──────────────────────┼──────────────────────────────────────────┤
│ No GROQ_API_KEY      │ Skip API entirely → keyword heuristic   │
│                      │ conf=0.6                                 │
├──────────────────────┼──────────────────────────────────────────┤
│ Network / HTTP error │ Retry up to 3×, exponential back-off    │
│                      │ (1s → 2s → 4s), then try fallback model │
│                      │ (llama-3.3-70b-versatile), then keyword    │
├──────────────────────┼──────────────────────────────────────────┤
│ Unexpected LLM resp  │ _parse_intent() tries:                  │
│                      │   1. exact match  → conf 1.0            │
│                      │   2. substring    → conf 0.7            │
│                      │   3. alias map    → conf 0.7            │
│                      │   4. out_of_scope → conf 0.4            │
├──────────────────────┼──────────────────────────────────────────┤
│ Empty / None input   │ Returns out_of_scope, conf 1.0          │
│                      │ immediately, no API call                │
└──────────────────────┴──────────────────────────────────────────┘

Model note
----------
The phase spec mentioned "gpt-oss-120b" — that is an internal OpenAI
model identifier, not available on Groq. Groq serves the Llama / Mixtral
family. We use:
  Primary  : llama-3.3-70b-versatile  (best quality, ~200 tok/s)
  Fallback : llama-3.3-70b-versatile     (fastest, lowest latency)
''')

## 7. Live Demo (requires GROQ_API_KEY)

In [ ]:
# Set your key: os.environ['GROQ_API_KEY'] = 'gsk_...'
# or export GROQ_API_KEY=gsk_... before launching Jupyter

demo_texts = [
    "I've been having really dark thoughts lately and don't know who to talk to.",
    "What's the best programming language to learn in 2025?",
    "Hey! Good to see you again.",
    "I need to head off now, thanks for everything.",
    "You've been incredibly helpful, I really mean it.",
]

live_clf = IntentClassifier()   # re-reads GROQ_API_KEY from env
print(f'{"Text":<65}  {"Intent":<34}  Conf  Mode')
print('-' * 120)
for text in demo_texts:
    r = live_clf.classify(text)
    mode_short = r['model_used'].replace('llama-3.3-70b-versatile','llama-70b').replace('keyword_fallback','kw-fallback')
    print(f"{text[:63]:<65}  {r['intent']:<34}  {r['confidence']:.2f}  {mode_short}")